In [1]:
import os
import re
import gc
import random
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report
)

from transformers import AutoTokenizer, AutoModelForCausalLM

print("Libraries imported successfully.")
print("PyTorch:", torch.__version__)
print("Transformers:", __import__("transformers").__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Total VRAM:",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
          "GB")

Libraries imported successfully.
PyTorch: 2.10.0+cu128
Transformers: 5.0.0
CUDA available: True
GPU: Tesla T4
Total VRAM: 14.56 GB


In [2]:
MODEL_ID = "google/gemma-3-4b-it"

DATA_DIR = "/kaggle/input/datasets/taufiqzahantushar/checksent-bn/CheckSent-BN"

TRAIN_PATH = os.path.join(DATA_DIR, "train_data.tsv")
TEST_PATH = os.path.join(DATA_DIR, "test_data.tsv")

MAX_NEW_TOKENS = 10

print("Model:", MODEL_ID)
print("Dataset directory:", DATA_DIR)
print("Max new tokens:", MAX_NEW_TOKENS)

Model: google/gemma-3-4b-it
Dataset directory: /kaggle/input/datasets/taufiqzahantushar/checksent-bn/CheckSent-BN
Max new tokens: 10


In [3]:
train_df = pd.read_csv(TRAIN_PATH, sep="\t")
test_df = pd.read_csv(TEST_PATH, sep="\t")

print("Train dataset loaded:", train_df.shape)
print("Test dataset loaded:", test_df.shape)

print("\nColumns:")
print(train_df.columns.tolist())

print("\nTrain sample:")
display(train_df.head())

print("\nTest sample:")
display(test_df.head())

Train dataset loaded: (9254, 6)
Test dataset loaded: (2314, 6)

Columns:
['ID', 'Date', 'News Portal', 'Headline', 'Claim', 'Sentiment']

Train sample:


,ID,Date,News Portal,Headline,Claim,Sentiment
0,BN/PRTD/241109/008,09-11-2024,Sangbad Pratidin,"দৈর্ঘ্য মাত্র ৪ ইঞ্চি, উপত্যকায় জঙ্গি দমনে সে...",Checkworthy,NEUTRAL
1,BN/BTMN/250103/009,03-01-2025,Bartaman,"বিএসএফের মদতে অনুপ্রবেশ: মমতা, বাংলাকে অশান্ত ...",Checkworthy,NEGATIVE
2,BN/PRTD/250111/008,11-01-2025,Sangbad Pratidin,"বেঙ্গালুরুকে হারিয়ে অঘটন মহামেডানের, ডার্বির আ...",Checkworthy,POSITIVE
3,BN/BTMN/250110/002,10-01-2025,Bartaman,কুম্ভমেলায় নিখোঁজ ব্যক্তিকে খুঁজতে এবার এআই প...,Checkworthy,NEUTRAL
4,BN/PRTD/241203/077,03-12-2024,Sangbad Pratidin,কেন বাংলাদেশের অবস্থায় প্রতিবাদী হব না?,Not-Checkworthy,NEUTRAL



Test sample:


,ID,Date,News Portal,Headline,Claim,Sentiment
0,BN/PRTD/241028/022,28-10-2024,Sangbad Pratidin,বেঙ্গালুরু-দমদম বিমানে বোমাতঙ্ক! সোশাল মিডিয়ায়...,Checkworthy,NEGATIVE
1,BN/PRTD/240823/074,23-08-2024,Sangbad Pratidin,"নেপালে নদীতে পড়ল ভারতীয় পর্যটকবোঝাই বাস, মৃত অ...",Checkworthy,NEGATIVE
2,BN/PRTD/241001/004,01-10-2024,Sangbad Pratidin,"পুজো হবে নির্বিঘ্নে, আশ্বাস ইউনুস সরকারের, মহা...",Checkworthy,NEUTRAL
3,BN/PRTD/241002/011,02-10-2024,Sangbad Pratidin,"হেঁশেলে হাতাহাতি, বউয়ের কান কাটল বর, পালটা ফাট...",Checkworthy,NEGATIVE
4,BN/PRTD/241127/029,27-11-2024,Sangbad Pratidin,"একজনের অধীনে তিন পরিবার, চিকিৎসা দিতে গ্রাম ‘দ...",Checkworthy,NEUTRAL


In [4]:
label_mapping = {
    "NEGATIVE": 0,
    "NEUTRAL": 1,
    "POSITIVE": 2
}

id_to_label = {
    0: "NEGATIVE",
    1: "NEUTRAL",
    2: "POSITIVE"
}

train_df["label_id"] = train_df["Sentiment"].map(label_mapping)
test_df["label_id"] = test_df["Sentiment"].map(label_mapping)

print("Sentiment labels prepared successfully.")

print("\nTraining distribution:")
print(train_df["Sentiment"].value_counts())

print("\nTest distribution:")
print(test_df["Sentiment"].value_counts())

print("\nMissing labels:")
print("Train:", train_df["label_id"].isna().sum())
print("Test:", test_df["label_id"].isna().sum())

Sentiment labels prepared successfully.

Training distribution:
Sentiment
NEGATIVE    4573
NEUTRAL     3187
POSITIVE    1494
Name: count, dtype: int64

Test distribution:
Sentiment
NEGATIVE    1123
NEUTRAL      825
POSITIVE     366
Name: count, dtype: int64

Missing labels:
Train: 0
Test: 0


In [5]:
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("tushartoken")

print("Hugging Face token loaded successfully.")

Hugging Face token loaded successfully.


In [6]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN
)

print("Tokenizer loaded successfully.")
print("Tokenizer class:", tokenizer.__class__.__name__)
print("Vocabulary size:", tokenizer.vocab_size)

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

Tokenizer loaded successfully.
Tokenizer class: GemmaTokenizer
Vocabulary size: 262144


In [7]:
print("Loading Gemma 3 4B...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

model.eval()

print("\nGemma 3 4B loaded successfully.")
print("Model device:", model.device)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading Gemma 3 4B...


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]


Gemma 3 4B loaded successfully.
Model device: cuda:0


In [8]:
test_headline = test_df.iloc[0]["Headline"]

prompt = f"""You are performing Bengali sentiment classification.

Choose exactly ONE label:
NEGATIVE
NEUTRAL
POSITIVE

Return ONLY ONE label.

Headline:
{test_headline}

Label:"""

inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

input_ids = inputs["input_ids"].to(model.device)

with torch.no_grad():
    outputs = model.generate(
        input_ids=input_ids,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False
    )

generated_tokens = outputs[0, input_ids.shape[1]:]

generated_text = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
).strip()

print("Headline:")
print(test_headline)

print("\nModel output:")
print(repr(generated_text))

print("\nActual label:")
print(test_df.iloc[0]["Sentiment"])

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Headline:
বেঙ্গালুরু-দমদম বিমানে বোমাতঙ্ক! সোশাল মিডিয়ায় লাগাতার হুমকি পোস্টে আতঙ্ক

Model output:
'NEGATIVE'

Actual label:
NEGATIVE


In [9]:
def build_zero_shot_prompt(headline):
    return f"""You are performing Bengali sentiment classification.

Choose exactly ONE label:

NEGATIVE = The headline expresses a negative sentiment.
NEUTRAL = The headline expresses a neutral or factual sentiment.
POSITIVE = The headline expresses a positive sentiment.

Return ONLY ONE label: NEGATIVE, NEUTRAL, or POSITIVE.

Headline:
{headline}

Label:"""


def parse_sentiment(output):
    output = output.strip().upper()

    if "NEGATIVE" in output:
        return "NEGATIVE"
    elif "NEUTRAL" in output:
        return "NEUTRAL"
    elif "POSITIVE" in output:
        return "POSITIVE"
    else:
        return None


def predict_sentiment(headline):
    prompt = build_zero_shot_prompt(headline)

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    input_ids = inputs["input_ids"].to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False
        )

    generated_tokens = outputs[0, input_ids.shape[1]:]

    generated_text = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    prediction = parse_sentiment(generated_text)

    return prediction, generated_text


print("Zero-shot prediction functions defined successfully.")

Zero-shot prediction functions defined successfully.


In [10]:
sample_indices = [0, 1, 2, 10, 20]

for idx in sample_indices:
    headline = test_df.iloc[idx]["Headline"]
    actual = test_df.iloc[idx]["Sentiment"]

    prediction, raw_output = predict_sentiment(headline)

    print(f"\nSample {idx}")
    print("Headline:", headline)
    print("Model output:", repr(raw_output))
    print("Predicted:", prediction)
    print("Actual:", actual)
    print("Correct:", prediction == actual)


Sample 0
Headline: বেঙ্গালুরু-দমদম বিমানে বোমাতঙ্ক! সোশাল মিডিয়ায় লাগাতার হুমকি পোস্টে আতঙ্ক
Model output: 'NEGATIVE'
Predicted: NEGATIVE
Actual: NEGATIVE
Correct: True

Sample 1
Headline: নেপালে নদীতে পড়ল ভারতীয় পর্যটকবোঝাই বাস, মৃত অন্তত ১৪
Model output: 'NEGATIVE'
Predicted: NEGATIVE
Actual: NEGATIVE
Correct: True

Sample 2
Headline: পুজো হবে নির্বিঘ্নে, আশ্বাস ইউনুস সরকারের, মহালয়ার প্রস্তুতি বাংলাদেশে
Model output: 'NEUTRAL'
Predicted: NEUTRAL
Actual: NEUTRAL
Correct: True

Sample 10
Headline: প্রবল বর্ষণে বিপর্যস্ত মেঘালয়, হড়পা বান ও ধসে মৃত ১০ জন
Model output: 'NEGATIVE'
Predicted: NEGATIVE
Actual: NEGATIVE
Correct: True

Sample 20
Headline: ইউটিউবে রোনাল্ডোর সাতকাহন, কোন সাত কারণে সিআর সেভেনের চ্যানেলে বুঁদ ভক্তরা?
Model output: '```text\nPOSITIVE\n```'
Predicted: POSITIVE
Actual: POSITIVE
Correct: True


In [11]:
from tqdm.auto import tqdm

zero_shot_predictions = []
zero_shot_raw_outputs = []

print(f"Running zero-shot evaluation on {len(test_df)} test samples...")

for headline in tqdm(
    test_df["Headline"].tolist(),
    desc="Zero-shot"
):
    prediction, raw_output = predict_sentiment(headline)

    zero_shot_predictions.append(
        prediction if prediction is not None else "UNKNOWN"
    )
    zero_shot_raw_outputs.append(raw_output)

print("\nZero-shot evaluation completed.")
print("Total predictions:", len(zero_shot_predictions))
print("Unknown predictions:", zero_shot_predictions.count("UNKNOWN"))

Running zero-shot evaluation on 2314 test samples...


Zero-shot:   0%|          | 0/2314 [00:00<?, ?it/s]


Zero-shot evaluation completed.
Total predictions: 2314
Unknown predictions: 3


In [12]:
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report
)

y_true = test_df["Sentiment"].tolist()
y_pred = zero_shot_predictions

labels = ["NEGATIVE", "NEUTRAL", "POSITIVE"]

# Overall metrics using macro averaging
accuracy = accuracy_score(y_true, y_pred)

precision, recall, f1, _ = precision_recall_fscore_support(
    y_true,
    y_pred,
    labels=labels,
    average="macro",
    zero_division=0
)

print("=" * 60)
print("ZERO-SHOT EVALUATION RESULTS")
print("=" * 60)

print(f"Total Samples       : {len(y_true)}")
print(f"Unknown Predictions : {y_pred.count('UNKNOWN')}")

print(f"\nAccuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-score  : {f1:.4f}")

print("\n" + "=" * 60)
print("CLASS-WISE RESULTS")
print("=" * 60)

print(
    classification_report(
        y_true,
        y_pred,
        labels=labels,
        digits=4,
        zero_division=0
    )
)

ZERO-SHOT EVALUATION RESULTS
Total Samples       : 2314
Unknown Predictions : 3

Accuracy  : 0.7532
Precision : 0.7161
Recall    : 0.7371
F1-score  : 0.7233

CLASS-WISE RESULTS
              precision    recall  f1-score   support

    NEGATIVE     0.8936    0.8005    0.8445      1123
     NEUTRAL     0.6905    0.7139    0.7020       825
    POSITIVE     0.5642    0.6967    0.6235       366

   micro avg     0.7542    0.7532    0.7537      2314
   macro avg     0.7161    0.7371    0.7233      2314
weighted avg     0.7691    0.7532    0.7588      2314



In [13]:
def build_cot_prompt(headline):
    return f"""You are performing Bengali sentiment classification.

Choose exactly ONE label:

NEGATIVE = The headline expresses a negative sentiment.
NEUTRAL = The headline expresses a neutral or factual sentiment.
POSITIVE = The headline expresses a positive sentiment.

Carefully analyze the headline's meaning, emotional polarity,
and overall sentiment before making your decision.

Do not provide your reasoning.
Return ONLY ONE label:
NEGATIVE, NEUTRAL, or POSITIVE.

Headline:
{headline}

Final label:"""


def predict_sentiment_cot(headline):
    prompt = build_cot_prompt(headline)

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    input_ids = inputs["input_ids"].to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False
        )

    generated_tokens = outputs[0, input_ids.shape[1]:]

    generated_text = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    prediction = parse_sentiment(generated_text)

    return prediction, generated_text


print("CoT prediction functions defined successfully.")

CoT prediction functions defined successfully.


In [14]:
sample_indices = [0, 1, 2, 10, 20]

for idx in sample_indices:
    headline = test_df.iloc[idx]["Headline"]
    actual = test_df.iloc[idx]["Sentiment"]

    prediction, raw_output = predict_sentiment_cot(headline)

    print(f"\nSample {idx}")
    print("Headline:", headline)
    print("Model output:", repr(raw_output))
    print("Predicted:", prediction)
    print("Actual:", actual)
    print("Correct:", prediction == actual)


Sample 0
Headline: বেঙ্গালুরু-দমদম বিমানে বোমাতঙ্ক! সোশাল মিডিয়ায় লাগাতার হুমকি পোস্টে আতঙ্ক
Model output: 'NEGATIVE'
Predicted: NEGATIVE
Actual: NEGATIVE
Correct: True

Sample 1
Headline: নেপালে নদীতে পড়ল ভারতীয় পর্যটকবোঝাই বাস, মৃত অন্তত ১৪
Model output: 'NEGATIVE'
Predicted: NEGATIVE
Actual: NEGATIVE
Correct: True

Sample 2
Headline: পুজো হবে নির্বিঘ্নে, আশ্বাস ইউনুস সরকারের, মহালয়ার প্রস্তুতি বাংলাদেশে
Model output: 'NEUTRAL'
Predicted: NEUTRAL
Actual: NEUTRAL
Correct: True

Sample 10
Headline: প্রবল বর্ষণে বিপর্যস্ত মেঘালয়, হড়পা বান ও ধসে মৃত ১০ জন
Model output: 'NEGATIVE'
Predicted: NEGATIVE
Actual: NEGATIVE
Correct: True

Sample 20
Headline: ইউটিউবে রোনাল্ডোর সাতকাহন, কোন সাত কারণে সিআর সেভেনের চ্যানেলে বুঁদ ভক্তরা?
Model output: 'POSITIVE'
Predicted: POSITIVE
Actual: POSITIVE
Correct: True


In [15]:
from tqdm.auto import tqdm

cot_predictions = []
cot_raw_outputs = []

print(f"Running CoT evaluation on {len(test_df)} test samples...")

for headline in tqdm(
    test_df["Headline"].tolist(),
    desc="CoT"
):
    prediction, raw_output = predict_sentiment_cot(headline)

    cot_predictions.append(
        prediction if prediction is not None else "UNKNOWN"
    )
    cot_raw_outputs.append(raw_output)

print("\nCoT evaluation completed.")
print("Total predictions:", len(cot_predictions))
print("Unknown predictions:", cot_predictions.count("UNKNOWN"))

Running CoT evaluation on 2314 test samples...


CoT:   0%|          | 0/2314 [00:00<?, ?it/s]


CoT evaluation completed.
Total predictions: 2314
Unknown predictions: 0


In [16]:
y_true = test_df["Sentiment"].tolist()
y_pred_cot = cot_predictions

labels = ["NEGATIVE", "NEUTRAL", "POSITIVE"]

# Overall metrics using macro averaging
cot_accuracy = accuracy_score(y_true, y_pred_cot)

cot_precision, cot_recall, cot_f1, _ = precision_recall_fscore_support(
    y_true,
    y_pred_cot,
    labels=labels,
    average="macro",
    zero_division=0
)

print("=" * 60)
print("CoT EVALUATION RESULTS")
print("=" * 60)

print(f"Total Samples       : {len(y_true)}")
print(f"Unknown Predictions : {y_pred_cot.count('UNKNOWN')}")

print(f"\nAccuracy  : {cot_accuracy:.4f}")
print(f"Precision : {cot_precision:.4f}")
print(f"Recall    : {cot_recall:.4f}")
print(f"F1-score  : {cot_f1:.4f}")

print("\n" + "=" * 60)
print("CLASS-WISE RESULTS")
print("=" * 60)

print(
    classification_report(
        y_true,
        y_pred_cot,
        labels=labels,
        digits=4,
        zero_division=0
    )
)

CoT EVALUATION RESULTS
Total Samples       : 2314
Unknown Predictions : 0

Accuracy  : 0.7615
Precision : 0.7394
Recall    : 0.7097
F1-score  : 0.7193

CLASS-WISE RESULTS
              precision    recall  f1-score   support

    NEGATIVE     0.7912    0.9314    0.8556      1123
     NEUTRAL     0.7360    0.6048    0.6640       825
    POSITIVE     0.6911    0.5929    0.6382       366

    accuracy                         0.7615      2314
   macro avg     0.7394    0.7097    0.7193      2314
weighted avg     0.7557    0.7615    0.7529      2314

